# How much should you put on a trade?

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/how-much-to-put-on-a-trade)

This notebook reproduces every figure and every number in the article.

## How to run it

| | |
| --- | --- |
| **Run it, nothing to install** | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/position-sizing.ipynb) |
| **Open it in VS Code** | [Open in VS Code](https://vscode.dev/github/Arthurnoo/out-of-sample/blob/main/notebooks/position-sizing.ipynb) |

Colab runs it in your browser. VS Code opens the real editor, also in your
browser, with nothing to install. Either way the next cell installs whatever is
missing, so you can simply **Run All**. The first run downloads prices and takes
a minute or two.

On your own machine instead: `git clone https://github.com/Arthurnoo/out-of-sample`

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option('display.width', 130)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

Everything worth changing is in the next cell. The two that matter most are
`SYMBOL` and the strategy parameters, because they change the mean and the
volatility of the returns, and those two numbers are what the whole piece
computes from.

| Knob | What it does | Try |
| --- | --- | --- |
| `SYMBOL` | the market the strategy runs on | `QQQ`, `GLD`, `BTC-USD` |
| `START` | first day of the sample | `2010-01-01` |
| `LOOKBACK`, `HOLDING` | the dip rule from episode 1 | `1, 5` |
| `LEVERS` | the position sizes to compare | add `3`, `6` |
| `WINDOWS_YEARS` | sample lengths in the uncertainty test | add `3` |
| `N_DRAWS` | random windows drawn per length | lower it to go faster |

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — this is the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────

SYMBOL   = 'SPY'
START    = '2000-01-01'

LOOKBACK = 2              # the rule calls it a dip if the last 2 days lost money
HOLDING  = 1              # and holds for 1 day

LEVERS        = [1, 2, 5, 8, 10]         # position sizes compared on real data
WINDOWS_YEARS = [1, 2, 5, 10, 20]        # sample lengths in the uncertainty test
N_DRAWS       = 2000                     # random windows per length

BARS_PER_YEAR = 252

## 1. The strategy we will size

This is the rule from the first episode, reproduced in a few lines. What it
does is not the point here. What matters is that it produces a series of daily
returns with a mean and a volatility, which is all the sizing arithmetic needs.

The weighting by `1/H` is the overlapping-portfolio construction: if a signal
fires every day and we hold for H days, we hold a fraction of capital per live
signal rather than ignoring signals while invested.

In [ ]:
def get_returns(symbol, start=START):
    h = yf.Ticker(symbol).history(start=start, auto_adjust=True)
    if h.index.tz is not None:
        h.index = h.index.tz_localize(None)
    return h['Close'].pct_change().dropna()


def dip_strategy(r, lookback=LOOKBACK, holding=HOLDING):
    """Renvoie le P&L quotidien de la regle 'acheter apres une baisse'."""
    x = r.to_numpy(dtype=float)
    fell = pd.Series(x).rolling(lookback).sum().to_numpy() < 0   # signal du jour
    raw = fell.astype(float)
    raw[:lookback] = 0.0
    w = pd.Series(raw).rolling(holding).mean().fillna(0.0).to_numpy()
    pos = np.zeros(len(x))
    pos[1:] = w[:-1]                       # decide a la cloture t, porte t -> t+1
    return pd.Series(pos * x, index=r.index)


r = get_returns(SYMBOL)
pnl = dip_strategy(r)

mu = pnl.mean()            # esperance par jour
sd = pnl.std(ddof=1)       # volatilite par jour
print(f"{len(pnl)} days, {pnl.index[0].date()} to {pnl.index[-1].date()}")
print(f"mean {mu*1e4:.2f} bp per day    volatility {sd*100:.2f}% per day")
print(f"Sharpe {mu/sd*np.sqrt(BARS_PER_YEAR):.2f}")

## 2. The arithmetic

Run the returns at leverage `L` and the expected return per trade is `L*mu`,
which rises forever. The capital does not follow, because returns compound.
Over many trades the growth rate of the account is:

```
g(L) = L*mu - L**2 * sd**2 / 2
```

The first term is the expectation, the second is what volatility costs, and the
second grows with the square of the position. Setting the derivative to zero
gives the leverage that maximises growth:

```
L* = mu / sd**2
```

The curve is a parabola through the origin, so growth comes back to zero at
exactly `2 * L*`, with every single trade still showing a positive
expectation.

In [ ]:
def growth(L, mu=mu, sd=sd):
    """Taux de croissance du capital, par jour, au levier L."""
    return L * mu - L**2 * sd**2 / 2


L_star = mu / sd**2
L_zero = 2 * L_star
print(f"L*               {L_star:.2f}x")
print(f"growth reaches 0 {L_zero:.2f}x")

grid = np.linspace(0, L_zero * 1.3, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, grid * mu * 100, '--', color='#8b98a6',
        label='what the expected return promises')
ax.plot(grid, growth(grid) * 100, color='#2d6fa8', lw=2,
        label='what the capital compounds at')
ax.axhline(0, color='#1a2430', lw=1)
ax.axvline(L_star, color='#0e8f5f', ls=':'); ax.axvline(L_zero, color='#c83c50', ls=':')
ax.set_xlabel('leverage'); ax.set_ylabel('per day (%)'); ax.legend()
ax.set_title('The expected return keeps rising. The capital does not.', loc='left')
plt.show()

## 3. What actually happens on the real series

The formula above is an approximation: it assumes the per-period return is
small. At high leverage it is not, and more importantly the formula has no
notion of **bankruptcy**. If a single day gives `1 + L*r <= 0` the account is
at zero and there is nothing left to compound, while the formula carries on
politely computing a negative growth rate.

So we compound the real returns day by day instead, and we stop the moment the
account is wiped out.

In [ ]:
def run_at(L, pnl=pnl):
    """Compose les rendements reels au levier L, en s'arretant a la faillite."""
    fac = 1 + L * pnl.to_numpy()          # facteur multiplicatif de chaque jour
    if np.any(fac <= 0):                  # un seul jour fatal suffit
        k = int(np.argmax(fac <= 0))
        eq = np.concatenate([np.cumprod(fac[:k]), np.zeros(len(fac) - k)])
        ruined_on = pnl.index[k].date()
    else:
        eq = np.cumprod(fac)
        ruined_on = None
    peak = np.maximum.accumulate(np.maximum(eq, 1e-12))
    years = len(fac) / BARS_PER_YEAR
    return {'lever': L, 'final': eq[-1], 'ruined_on': ruined_on,
            'cagr_pct': (eq[-1]**(1/years) - 1)*100 if eq[-1] > 0 else -100.0,
            'max_dd_pct': (np.min(eq/peak) - 1)*100,
            'equity': pd.Series(np.where(eq > 0, eq, np.nan), index=pnl.index)}


res = [run_at(L) for L in LEVERS]
print(pd.DataFrame([{k: v for k, v in d.items() if k != 'equity'} for d in res])
      .to_string(index=False, float_format=lambda v: f'{v:11.2f}'))

worst = pnl.min()
print(f"\nWorst single day: {worst*100:.2f}% on {pnl.index[pnl.argmin()].date()}")
print(f"From {-1/worst:.1f}x, that one day alone takes the account to zero.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ramp = ['#bcd3e6', '#8fb5d4', '#5f92bf', '#2d6fa8', '#1b4a72']
for i, d in enumerate(res):
    ax.plot(d['equity'].index, d['equity'].values, lw=1.6,
            color=ramp[i % len(ramp)], label=f"{d['lever']:g}x")
ax.set_yscale('log')
ax.set_ylabel('capital, starting at 1 (log scale)'); ax.legend()
ax.set_title('The same strategy, five position sizes', loc='left')
plt.show()

## 4. The number you cannot measure

`L* = mu / sd**2` needs `mu`. Volatility is easy to estimate from a few weeks
of data. Expected return is not, and it is the harder of the two by a wide
margin.

To see how bad it is, we draw many random windows out of the real return
series, compute `L*` on each one, and look at the spread. Reading the table:
`p05` and `p95` bracket 90% of the estimates, and `above_2x` is the share of
windows that recommend a leverage past the point where growth turns
negative.

In [ ]:
rng = np.random.default_rng(0)
x = pnl.to_numpy()
truth = x.mean() / x.std(ddof=1)**2

rows = []
for y in WINDOWS_YEARS:
    n = int(y * BARS_PER_YEAR)
    if n >= len(x):
        continue
    starts = rng.integers(0, len(x) - n, size=N_DRAWS)
    est = np.array([x[s:s+n].mean() / x[s:s+n].std(ddof=1)**2 for s in starts])
    rows.append({'years': y, 'median': np.median(est),
                 'p05': np.percentile(est, 5), 'p95': np.percentile(est, 95),
                 'above_2x_pct': (est > 2*truth).mean()*100,
                 'negative_pct': (est < 0).mean()*100})

spread = pd.DataFrame(rows)
print(f"L* on the full sample: {truth:.2f}x,  growth turns negative at {2*truth:.2f}x\n")
print(spread.to_string(index=False, float_format=lambda v: f'{v:12.2f}'))

## 5. Fractional sizing

The parabola is flat near its top. Cutting the position in half therefore costs
little growth and removes a lot of risk, and that asymmetry is the practical
answer to the uncertainty in section 4: you do not need to know `mu` well if
you are running at a fraction of what it recommends.

In [ ]:
frac = []
for f, lab in [(1.0, 'full'), (0.5, 'half'), (0.25, 'quarter')]:
    d = run_at(f * L_star)
    frac.append({'rule': lab, 'lever': f*L_star, 'cagr_pct': d['cagr_pct'],
                 'max_dd_pct': d['max_dd_pct'], 'final': d['final']})
frac = pd.DataFrame(frac)
print(frac.to_string(index=False, float_format=lambda v: f'{v:11.2f}'))
print(f"\nHalf keeps {frac.cagr_pct[1]/frac.cagr_pct[0]*100:.0f}% of the return, "
      f"quarter keeps {frac.cagr_pct[2]/frac.cagr_pct[0]*100:.0f}%.")

## 6. Five things worth changing

1. **Set `SYMBOL = 'BTC-USD'`.** Volatility roughly triples, and `L*` falls
   with the square of it. Check whether the optimum lands below 1x, which would
   mean the honest position is a fraction of your capital.
2. **Change `LOOKBACK` and `HOLDING`.** A different cell of the episode 1 grid
   has a different mean and volatility, so it has a different `L*`. Does the
   cell with the best Sharpe also have the best growth at its own `L*`?
3. **Add a transaction cost** to `dip_strategy`. Costs cut `mu` without
   touching `sd`, so `L*` falls in direct proportion. At what cost does the
   optimal position drop below 1x?
4. **Lower `WINDOWS_YEARS` to `[0.25, 0.5, 1]`.** How short does the sample
   have to be before the 90% band stops excluding zero entirely?
5. **Replace the random windows with sequential ones**, so each estimate uses
   only data that preceded it, then size the next period with it. That is the
   honest version, and it will be worse than what section 4 shows.

Found something I got wrong? There is a form at the bottom of the article.